# 📦 FakeInversion - Notebook 1: Setup & Data Preparation

This notebook handles:
1. Environment setup (install dependencies, check GPU)
2. Mount Google Drive
3. Build prompts from HuggingFace dataset
4. Generate fake images with SD-1.5
5. Fetch real images from URLs
6. Prepare train/val/test splits

In [ ]:
# Step 0: Clone project to Colab (if running from GitHub)
# Skip this if you uploaded the project manually

import os

PROJECT_DIR = '/content/fake_inversion'

if not os.path.exists(PROJECT_DIR):
    # Upload your project files here or clone from GitHub
    print('Please upload your fake_inversion project to /content/fake_inversion')
    print('You can do this by:')
    print('  1. Zip your project and upload via Colab file browser')
    print('  2. Or mount Google Drive and copy from there')
else:
    print(f'Project found at {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, PROJECT_DIR)

In [ ]:
# Step 1: Install dependencies
!pip install -q torch torchvision diffusers transformers accelerate datasets \
    Pillow numpy scipy scikit-learn pandas matplotlib seaborn tqdm tensorboard \
    requests safetensors

In [ ]:
# Step 2: Check environment
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
# Step 3: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set up project directory on Drive
import config
config.ensure_dirs()
config.print_config()

In [ ]:
# Step 4: Build prompts
from data.build_prompts import build_prompts, load_prompts

# Build subset of 500 prompts
prompts = build_prompts(num_prompts=500, subset=True)
print(f'\nSample prompts:')
for i, p in enumerate(prompts[:5]):
    print(f'  {i}: {p[:100]}...')

In [ ]:
# Step 5: Generate fake images (start with SD-1.5 only for training)
from data.generate_fake_images import generate_with_sd_pipeline

prompts = load_prompts()

# Generate SD-1.5 fakes (this is what we train on)
generate_with_sd_pipeline(
    model_name='sd-15',
    prompts=prompts,
    save_dir=os.path.join(config.FAKE_IMAGES_DIR, 'sd-15'),
    num_images=config.SUBSET_SIZE,
)

In [ ]:
# Step 6: Fetch real images from JSONL URLs
from data.fetch_real_images import fetch_from_jsonl, fetch_all_real_images

# Fetch real images for SD-1.5 (for training)
fetch_from_jsonl(
    jsonl_path=os.path.join(config.URL_FILES_DIR, 'sd-15.jsonl'),
    save_dir=os.path.join(config.REAL_IMAGES_DIR, 'sd-15'),
    max_images=config.SUBSET_SIZE,
)

# Optionally: fetch real images for other models too (for evaluation)
# fetch_all_real_images(max_images_per_source=config.SUBSET_SIZE)

In [ ]:
# Step 7: Prepare dataset splits
from data.prepare_dataset import prepare_all

prepare_all(subset=True)

# Check manifests
import glob
for f in sorted(glob.glob(os.path.join(config.MANIFESTS_DIR, '*.csv'))):
    with open(f) as fh:
        lines = fh.readlines()
    print(f'{os.path.basename(f):30s}: {len(lines)-1} entries')

## ✅ Data preparation complete!

Next: Run **Notebook 02** for DDIM Inversion feature extraction.